# 12. A/B Sequential Experiment：怎样逐日看数据，又不把第一天噪声当胜利？

## 面试回答主线

固定样本 z-test 的 0.05 阈值只适用于预先约定的一次分析；每天反复查看并在首次显著时停止，会膨胀整体假阳性。最简单的安全方案是预先声明 looks，用 Bonferroni alpha spending 将总 alpha 分到每天，并同时设置最小样本量与最小业务提升。面试时我会对十四天真实转化计数逐日累计，输出 conversion、lift、pooled SE、z、nominal p 和 sequential boundary。这个案例第一天恰好越过 1.96，但随后回落；只有第十四天才越过更严格边界，效果估计也从夸大的 10pp 回到 3.7pp。还必须先检查 Sample Ratio Mismatch，否则分流故障下的显著性没有解释意义。生产可采用 group sequential、always-valid p-value 或 Bayesian 设计，但规则必须在看数据前固定。

## 1. 真实案例：十四天控制组与实验组转化计数

每天每组 100 次有效访问。第一天实验组偶然有 20 次转化，后续逐渐稳定在约 14%；控制组约 10.5%。逐日数据保留日期、流量和转化，便于完整审计。

In [1]:
from pprint import pprint  # 导入结构化打印工具展示逐日实验轨迹
import math  # 导入平方根函数手写比例 z-test
from statistics import NormalDist  # 导入标准正态分布计算 p-value 与临界值
control_conversions = [10, 12, 9, 11, 10, 13, 8, 12, 9, 11, 10, 12, 9, 11]  # 定义十四天控制组真实转化数
treatment_conversions = [20, 10, 11, 13, 14, 15, 13, 15, 14, 15, 14, 16, 14, 15]  # 定义实验组首日波动后逐渐稳定的转化数
daily_visitors = 100  # 设置每天每组相同的有效访问数
days = [{"day": index + 1, "control_n": daily_visitors, "control_y": control_conversions[index], "treatment_n": daily_visitors, "treatment_y": treatment_conversions[index]} for index in range(len(control_conversions))]  # 组装十四天可复现 A/B 事件
preview = [{"天": row["day"], "控制转化": f'{row["control_y"]}/{row["control_n"]}', "实验转化": f'{row["treatment_y"]}/{row["treatment_n"]}', "当日差值pp": round((row["treatment_y"] / row["treatment_n"] - row["control_y"] / row["control_n"]) * 100, 1)} for row in days]  # 汇总逐日业务输入
print("A/B 实验每日数据：")  # 输出真实案例标题
pprint(preview, sort_dicts=False)  # 展示第一天尖峰与后续稳定效果

A/B 实验每日数据：
[{'天': 1, '控制转化': '10/100', '实验转化': '20/100', '当日差值pp': 10.0},
 {'天': 2, '控制转化': '12/100', '实验转化': '10/100', '当日差值pp': -2.0},
 {'天': 3, '控制转化': '9/100', '实验转化': '11/100', '当日差值pp': 2.0},
 {'天': 4, '控制转化': '11/100', '实验转化': '13/100', '当日差值pp': 2.0},
 {'天': 5, '控制转化': '10/100', '实验转化': '14/100', '当日差值pp': 4.0},
 {'天': 6, '控制转化': '13/100', '实验转化': '15/100', '当日差值pp': 2.0},
 {'天': 7, '控制转化': '8/100', '实验转化': '13/100', '当日差值pp': 5.0},
 {'天': 8, '控制转化': '12/100', '实验转化': '15/100', '当日差值pp': 3.0},
 {'天': 9, '控制转化': '9/100', '实验转化': '14/100', '当日差值pp': 5.0},
 {'天': 10, '控制转化': '11/100', '实验转化': '15/100', '当日差值pp': 4.0},
 {'天': 11, '控制转化': '10/100', '实验转化': '14/100', '当日差值pp': 4.0},
 {'天': 12, '控制转化': '12/100', '实验转化': '16/100', '当日差值pp': 4.0},
 {'天': 13, '控制转化': '9/100', '实验转化': '14/100', '当日差值pp': 5.0},
 {'天': 14, '控制转化': '11/100', '实验转化': '15/100', '当日差值pp': 4.0}]


## 2. Baseline（基线）：每天用固定 1.96 门槛偷看并立即停试

下面手写累计两比例 pooled z-test。错误基线每天都用 nominal α=0.05 双侧门槛，第一天 z≈1.98 就宣布胜利，报告实验提升 10 个百分点。

In [2]:
normal = NormalDist()  # 创建标准正态分布对象计算双侧显著性
def two_proportion_stats(control_y, control_n, treatment_y, treatment_n):  # 手写累计两样本比例 z-test
    control_rate = control_y / control_n  # 计算控制组累计转化率
    treatment_rate = treatment_y / treatment_n  # 计算实验组累计转化率
    pooled_rate = (control_y + treatment_y) / (control_n + treatment_n)  # 在零假设下计算 pooled conversion
    standard_error = math.sqrt(pooled_rate * (1 - pooled_rate) * (1 / control_n + 1 / treatment_n))  # 计算两比例差的 pooled 标准误
    z_score = (treatment_rate - control_rate) / standard_error if standard_error else 0.0  # 将累计提升标准化为 z 统计量
    p_value = 2 * (1 - normal.cdf(abs(z_score)))  # 计算双侧 nominal p-value
    return control_rate, treatment_rate, treatment_rate - control_rate, standard_error, z_score, p_value  # 返回逐日审计所需全部中间量
nominal_boundary = normal.inv_cdf(1 - 0.05 / 2)  # 计算单次固定样本双侧 α=0.05 临界值
baseline_ledger = []  # 收集每天偷看的累计统计
control_total = 0  # 初始化控制组累计转化数
treatment_total = 0  # 初始化实验组累计转化数
for row in days:  # 按日期顺序模拟每天分析一次
    control_total += row["control_y"]  # 累加控制组转化
    treatment_total += row["treatment_y"]  # 累加实验组转化
    control_n = row["day"] * daily_visitors  # 计算截至当天控制组样本数
    treatment_n = row["day"] * daily_visitors  # 计算截至当天实验组样本数
    control_rate, treatment_rate, lift, standard_error, z_score, p_value = two_proportion_stats(control_total, control_n, treatment_total, treatment_n)  # 计算当前 look 的完整统计量
    baseline_ledger.append({"day": row["day"], "control_rate": control_rate, "treatment_rate": treatment_rate, "lift": lift, "SE": standard_error, "z": z_score, "nominal_p": p_value, "nominal显著": abs(z_score) >= nominal_boundary})  # 保存每日固定阈值判断
baseline_stop = next(row for row in baseline_ledger if row["nominal显著"])  # 复现首次 nominal 显著立即停止的错误策略
print("固定阈值逐日统计：")  # 标注当前输出属于 peeking 基线
pprint([{**row, "control_rate": round(row["control_rate"], 4), "treatment_rate": round(row["treatment_rate"], 4), "lift": round(row["lift"], 4), "SE": round(row["SE"], 4), "z": round(row["z"], 3), "nominal_p": round(row["nominal_p"], 4)} for row in baseline_ledger], sort_dicts=False)  # 展示显著性如何出现、消失再出现
print({"错误停试日": baseline_stop["day"], "当时lift百分点": round(baseline_stop["lift"] * 100, 2), "nominal边界": round(nominal_boundary, 3)})  # 展示第一天噪声造成的夸大结论

固定阈值逐日统计：
[{'day': 1,
  'control_rate': 0.1,
  'treatment_rate': 0.2,
  'lift': 0.1,
  'SE': 0.0505,
  'z': 1.98,
  'nominal_p': 0.0477,
  'nominal显著': True},
 {'day': 2,
  'control_rate': 0.11,
  'treatment_rate': 0.15,
  'lift': 0.04,
  'SE': 0.0336,
  'z': 1.189,
  'nominal_p': 0.2343,
  'nominal显著': False},
 {'day': 3,
  'control_rate': 0.1033,
  'treatment_rate': 0.1367,
  'lift': 0.0333,
  'SE': 0.0265,
  'z': 1.256,
  'nominal_p': 0.209,
  'nominal显著': False},
 {'day': 4,
  'control_rate': 0.105,
  'treatment_rate': 0.135,
  'lift': 0.03,
  'SE': 0.023,
  'z': 1.306,
  'nominal_p': 0.1917,
  'nominal显著': False},
 {'day': 5,
  'control_rate': 0.104,
  'treatment_rate': 0.136,
  'lift': 0.032,
  'SE': 0.0206,
  'z': 1.557,
  'nominal_p': 0.1195,
  'nominal显著': False},
 {'day': 6,
  'control_rate': 0.1083,
  'treatment_rate': 0.1383,
  'lift': 0.03,
  'SE': 0.019,
  'z': 1.58,
  'nominal_p': 0.1141,
  'nominal显著': False},
 {'day': 7,
  'control_rate': 0.1043,
  'treatment_rate': 0.

## 3. 核心机制：预先声明十四次 look 并分配总 alpha

Bonferroni 是保守但透明的 alpha spending：每天使用 `0.05/14`，双侧 z 边界约 2.914。规则还要求每组至少 1000 人且 lift 至少 2pp，避免统计显著但业务无意义。

In [3]:
total_alpha = 0.05  # 设置整个实验允许的 family-wise error rate
planned_looks = len(days)  # 在看数据前声明最多分析十四次
alpha_per_look = total_alpha / planned_looks  # 用 Bonferroni 将总 alpha 平均分配到每次 look
sequential_boundary = normal.inv_cdf(1 - alpha_per_look / 2)  # 计算每天共同使用的双侧序贯 z 门槛
minimum_sample_per_arm = 1000  # 设置停试前每组至少积累一千次访问
minimum_practical_lift = 0.02  # 设置业务上值得发布的最小绝对提升两百分点
print({"总alpha": total_alpha, "计划look数": planned_looks, "每次alpha": round(alpha_per_look, 6), "序贯z边界": round(sequential_boundary, 4), "最小每组样本": minimum_sample_per_arm, "最小业务lift": minimum_practical_lift})  # 展示预注册的统计与业务门槛

{'总alpha': 0.05, '计划look数': 14, '每次alpha': 0.003571, '序贯z边界': 2.9137, '最小每组样本': 1000, '最小业务lift': 0.02}


## 4. 逐日 Sequential Gate：统计、样本和业务门槛同时通过

复用相同累计数据，不重新挑天。每天记录 z 与严格边界的距离，并分别标记 sample、practical、statistical 三个 gate；只有全部为真才能停试。

In [4]:
sequential_ledger = []  # 收集十四次预注册 look 的门禁结果
for row in baseline_ledger:  # 复用同一累计统计避免改变样本或公式
    sample_gate = row["day"] * daily_visitors >= minimum_sample_per_arm  # 检查每组样本数是否达到最低要求
    practical_gate = row["lift"] >= minimum_practical_lift  # 检查绝对提升是否达到业务价值门槛
    statistical_gate = row["z"] >= sequential_boundary  # 检查正向效果是否越过序贯统计边界
    stop = sample_gate and practical_gate and statistical_gate  # 只有三项共同通过才允许宣布实验胜利
    sequential_ledger.append({"day": row["day"], "累计每组N": row["day"] * daily_visitors, "lift_pp": row["lift"] * 100, "z": row["z"], "距离边界": row["z"] - sequential_boundary, "样本门": sample_gate, "业务门": practical_gate, "统计门": statistical_gate, "允许停试": stop})  # 保存逐日完整门禁账本
sequential_stop = next((row for row in sequential_ledger if row["允许停试"]), None)  # 找到第一个合法停试日或继续实验
print("Sequential Experiment 逐日门禁：")  # 输出核心方案轨迹标题
pprint([{**row, "lift_pp": round(row["lift_pp"], 2), "z": round(row["z"], 3), "距离边界": round(row["距离边界"], 3)} for row in sequential_ledger], sort_dicts=False)  # 展示每天距离统计和业务门槛还有多远

Sequential Experiment 逐日门禁：
[{'day': 1,
  '累计每组N': 100,
  'lift_pp': 10.0,
  'z': 1.98,
  '距离边界': -0.933,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 2,
  '累计每组N': 200,
  'lift_pp': 4.0,
  'z': 1.189,
  '距离边界': -1.724,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 3,
  '累计每组N': 300,
  'lift_pp': 3.33,
  'z': 1.256,
  '距离边界': -1.657,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 4,
  '累计每组N': 400,
  'lift_pp': 3.0,
  'z': 1.306,
  '距离边界': -1.608,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 5,
  '累计每组N': 500,
  'lift_pp': 3.2,
  'z': 1.557,
  '距离边界': -1.357,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 6,
  '累计每组N': 600,
  'lift_pp': 3.0,
  'z': 1.58,
  '距离边界': -1.333,
  '样本门': False,
  '业务门': True,
  '统计门': False,
  '允许停试': False},
 {'day': 7,
  '累计每组N': 700,
  'lift_pp': 3.29,
  'z': 1.887,
  '距离边界': -1.027,
  '样本门': False,
  '业务门': True,
  '统

## 5. 结果解读：第一天 10pp 是噪声，第十四天 3.71pp 才可发布

固定阈值第一天停试后看不到第二天 z 回落到 1.19。预注册规则一直运行到第十四天，估计更稳定且刚刚越过严格边界；逐日 ledger 保留所有未通过日，避免只展示胜利时刻。

In [5]:
final_row = baseline_ledger[-1]  # 读取完整十四天实验的最终累计统计
result_summary = {"peeking停试日": baseline_stop["day"], "peeking估计lift_pp": baseline_stop["lift"] * 100, "sequential停试日": sequential_stop["day"] if sequential_stop else None, "sequential估计lift_pp": sequential_stop["lift_pp"] if sequential_stop else None, "最终控制转化率": final_row["control_rate"], "最终实验转化率": final_row["treatment_rate"], "最终z": final_row["z"], "严格边界": sequential_boundary}  # 汇总两种停试策略的结论差异
print("Peeking 与预注册序贯结果：")  # 输出结果解读标题
pprint({key: round(value, 4) if isinstance(value, float) else value for key, value in result_summary.items()}, sort_dicts=False)  # 展示夸大早停与稳定最终估计

Peeking 与预注册序贯结果：
{'peeking停试日': 1,
 'peeking估计lift_pp': 10.0,
 'sequential停试日': 14,
 'sequential估计lift_pp': 3.7143,
 '最终控制转化率': 0.105,
 '最终实验转化率': 0.1421,
 '最终z': 2.9861,
 '严格边界': 2.9137}


## 6. 失败案例与修正：Sample Ratio Mismatch 让显著性失去解释

假设路由 bug 让实验组累计 1800 人、控制组 1400 人，而预期 50/50。SRM z 统计远超门槛，说明随机分流或埋点失效；此时不能用转化 p-value 决策。修正是先阻断实验结论、排查分桶和曝光日志，修复后重新预注册。

In [6]:
bad_control_n = 1400  # 设置分流故障后的控制组累计人数
bad_treatment_n = 1800  # 设置明显超过预期的实验组累计人数
bad_total_n = bad_control_n + bad_treatment_n  # 计算分流故障后的总样本数
expected_per_arm = bad_total_n / 2  # 根据预注册五五分流计算每组期望人数
srm_standard_deviation = math.sqrt(bad_total_n * 0.5 * 0.5)  # 计算二项分流下实验组人数标准差
srm_z = (bad_treatment_n - expected_per_arm) / srm_standard_deviation  # 将实际实验组人数偏差标准化
srm_p = 2 * (1 - normal.cdf(abs(srm_z)))  # 计算双侧 Sample Ratio Mismatch p-value
srm_failed = srm_p < 0.001  # 用严格门槛识别分流或曝光记录故障
experiment_decision = "INVALIDATE_AND_DEBUG_ASSIGNMENT" if srm_failed else "CONTINUE_STATISTICAL_GATE"  # SRM 失败时阻断任何效果结论
print({"控制N": bad_control_n, "实验N": bad_treatment_n, "期望每组": expected_per_arm, "SRM_z": round(srm_z, 3), "SRM_p": srm_p, "门禁失败": srm_failed, "修正决策": experiment_decision})  # 展示分流故障与前置门禁

{'控制N': 1400, '实验N': 1800, '期望每组': 1600.0, 'SRM_z': 7.071, 'SRM_p': 1.5374368445009168e-12, '门禁失败': True, '修正决策': 'INVALIDATE_AND_DEBUG_ASSIGNMENT'}


## 7. 生产差距与最小回归检查

生产实验应预注册 primary metric、guardrail、分层随机化、分析窗口和停试规则，并处理机器人、重复用户、延迟转化与新奇效应。Bonferroni 保守但透明，更高效设计可用 O'Brien-Fleming、alpha spending function 或 always-valid confidence sequence。下面的断言只验证本实验的逐日统计、peeking、严格边界、业务门和 SRM。

In [7]:
assert len(days) >= 6  # 确认真实逐日实验数据数量满足教学要求
assert baseline_stop["day"] == 1  # 确认固定阈值偷看会在首日噪声处错误停试
assert sequential_stop is not None and sequential_stop["day"] == 14  # 确认预注册门禁只在第十四天全部通过
assert sequential_boundary > nominal_boundary  # 确认多次查看使用比单次检验更严格的统计边界
assert baseline_stop["lift"] > sequential_stop["lift_pp"] / 100  # 确认首日停试明显夸大真实稳定提升
assert srm_failed is True and experiment_decision == "INVALIDATE_AND_DEBUG_ASSIGNMENT"  # 确认分流故障优先阻断效果判断
print("回归检查通过：逐日 z 统计、alpha spending、业务门槛与 SRM 前置门禁均已验证。")  # 输出最终验收结论

回归检查通过：逐日 z 统计、alpha spending、业务门槛与 SRM 前置门禁均已验证。
